In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Hello world

<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vytvořen s použitím následujících požadavků.
Doporučujeme používat tyto verze nebo novější.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Tento příklad se skládá ze dvou částí. Nejprve vytvoříš jednoduchý kvantový program a spustíš ho na kvantové procesorové jednotce (QPU). Protože skutečný kvantový výzkum vyžaduje mnohem robustnější programy, ve druhé části ([Škálování na velký počet qubitů](#scale-to-large-numbers-of-qubits)) tento jednoduchý program rozšíříš na úroveň praktického použití.

## Instalace a ověření
1. Pokud jsi ještě nenainstaloval(a) Qiskit, najdi pokyny v průvodci [Rychlý start](/guides/quick-start).

    - Nainstaluj Qiskit Runtime pro spouštění úloh na kvantovém hardwaru:

        ```bash
        pip install qiskit-ibm-runtime
        ```

    - Nastav prostředí pro lokální spouštění Jupyter notebooků:

        ```bash
        pip install jupyter
        ```

2. Nastav autentizaci pro přístup ke kvantovému hardwaru přes bezplatný [Open Plan](/guides/plans-overview#open-plan).

    (Pokud jsi obdržel(a) e-mailové pozvání k připojení k účtu, postupuj místo toho podle [kroků pro pozvané uživatele](/guides/cloud-setup-invited).)

    - Přihlas se nebo si vytvoř účet na [IBM Quantum Platform](https://quantum.cloud.ibm.com/).
         > **Note:** Pokud se připojuješ přes proxy server, musíš používat Qiskit Runtime verze 0.44.0 nebo novější.
    - Vygeneruj svůj API klíč (označovaný také jako *API token*) na [řídicím panelu](https://quantum.cloud.ibm.com/) a ulož ho na bezpečné místo.
    - Přejdi na stránku [Instances](https://quantum.cloud.ibm.com/instances) a najdi instanci, kterou chceš použít. Přejeď myší nad jejím CRN a kliknutím ho zkopíruj.

    - Ulož přihlašovací údaje lokálně pomocí tohoto kódu:

        ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        QiskitRuntimeService.save_account(
        token="<your-api-key>", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
        instance="<CRN>", # Optional
        )
        ```

3. Tento Python kód pak můžeš použít kdykoli, když se chceš ověřit vůči Qiskit Runtime Service:
    ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        # Run every time you need the service
        service = QiskitRuntimeService()
    ```
> **Info:** Pokud používáš veřejný počítač nebo jiné nezabezpečené prostředí, postupuj místo toho podle [pokynů pro ruční autentizaci](/guides/cloud-setup-untrusted), aby byly tvoje přihlašovací údaje v bezpečí.
## Vytvoření a spuštění jednoduchého kvantového programu
Čtyři kroky k napsání kvantového programu pomocí vzorů Qiskit jsou:

1.  Namapuj problém do kvantového nativního formátu.

2.  Optimalizuj Circuit a operátory.

3.  Proveď výpočet pomocí primitivní kvantové funkce.

4.  Analyzuj výsledky.

### Krok 1. Namapuj problém do kvantového nativního formátu
V kvantovém programu jsou *kvantové Circuit* nativním formátem pro reprezentaci kvantových instrukcí a *operátory* představují pozorovatelné veličiny, které se mají měřit. Při vytváření Circuit zpravidla vytvoříš nový objekt [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) a pak do něj postupně přidáváš instrukce.
Následující kód vytváří Circuit, který produkuje *Bellův stav* — stav, ve kterém jsou dva Qubit plně provázány.

> **Note:** Qiskit SDK používá číslování bitů LSb 0, kde $n$-tá číslice má hodnotu $1 \ll n$ nebo $2^n$. Více informací najdeš v tématu [Pořadí bitů v Qiskit SDK](/guides/bit-ordering).

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from matplotlib import pyplot as plt
# Uncomment the next line if you want to use a simulator:
# from qiskit_ibm_runtime.fake_provider import FakeBelemV2


# Create a new circuit with two qubits
qc = QuantumCircuit(2)

# Add a Hadamard gate to qubit 0
qc.h(0)

# Perform a controlled-X gate on qubit 1, controlled by qubit 0
qc.cx(0, 1)

# Return a drawing of the circuit using MatPlotLib ("mpl").
# These guides are written by using Jupyter notebooks, which
# display the output of the last line of each cell.
# If you're running this in a script, use `print(qc.draw())` to
# print a text drawing.
qc.draw("mpl")

<Image src="../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg)

Všechny dostupné operace najdeš v dokumentaci k [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class).
Při vytváření kvantových Circuit musíš také zvážit, jaký typ dat chceš po provedení vrátit. Qiskit poskytuje dva způsoby vrácení dat: můžeš získat rozdělení pravděpodobnosti pro sadu Qubit, které se rozhodneš měřit, nebo můžeš získat střední hodnotu pozorovatelné veličiny. Připrav svoji úlohu k měření Circuit jedním z těchto dvou způsobů pomocí [Qiskit primitiv](/guides/get-started-with-primitives) (podrobně vysvětleno v [Kroku 3](#step-3-execute-using-the-quantum-primitives)).

Tento příklad měří střední hodnoty pomocí submodulu `qiskit.quantum_info`, který se specifikuje pomocí operátorů (matematické objekty používané k reprezentaci akce nebo procesu, který mění kvantový stav). Následující kód vytváří šest dvouqubitových Pauliho operátorů: `IZ`, `IX`, `ZI`, `XI`, `ZZ` a `XX`.

In [2]:
# Set up six different observables.

observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

> **Note:** Zde je například operátor `ZZ` zkratkou pro tenzorový součin $Z\otimes Z$, což znamená měření Z na Qubit 1 a Z na Qubit 0 dohromady a získání informace o korelaci mezi Qubit 1 a Qubit 0. Střední hodnoty tohoto typu se také typicky zapisují jako $\langle Z_1 Z_0 \rangle$.
> 
> Pokud je stav provázaný, pak by se měření $\langle Z_1 Z_0 \rangle$ mělo lišit od měření $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$. Pro konkrétní provázaný stav vytvořený naším výše popsaným Circuit by mělo být měření $\langle Z_1 Z_0 \rangle$ rovno 1 a měření $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$ rovno nule.
<span id="optimize"></span>

### Krok 2. Optimalizuj Circuit a operátory

Při provádění Circuit na zařízení je důležité optimalizovat sadu instrukcí, které Circuit obsahuje, a minimalizovat celkovou hloubku (přibližně počet instrukcí) Circuit. Tím zajistíš co nejlepší výsledky snížením vlivu chyb a šumu. Instrukce Circuit navíc musí odpovídat [Instruction Set Architecture (ISA)](/guides/transpile#instruction-set-architecture) Backend zařízení a musí zohledňovat základní Gate a propojení Qubit daného zařízení.

Následující kód vytvoří instanci skutečného zařízení, na které se odešle úloha, a transformuje Circuit a pozorovatelné veličiny tak, aby odpovídaly ISA daného Backend. Vyžaduje, aby sis již [uložil(a) přihlašovací údaje](/guides/cloud-setup).

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(simulator=False, operational=True)

# Convert to an ISA circuit and layout-mapped observables.
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

isa_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg)

### Krok 3. Proveď výpočet pomocí kvantových primitiv
Kvantové počítače mohou produkovat náhodné výsledky, takže obvykle sbíráš vzorek výstupů opakovaným spuštěním Circuit. Střední hodnotu pozorovatelné veličiny můžeš odhadnout pomocí třídy `Estimator`. `Estimator` je jedním ze dvou [primitiv](/guides/get-started-with-primitives); druhým je `Sampler`, který lze použít k získání dat z kvantového počítače. Tyto objekty disponují metodou `run()`, která provede výběr Circuit, pozorovatelných veličin a parametrů (pokud jsou k dispozici) pomocí [primitive unified bloc (PUB).](/guides/primitives#sampler)

In [4]:
# Construct the Estimator instance.

estimator = Estimator(mode=backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 5000

mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

# One pub, with one circuit to run against five different observables.
job = estimator.run([(isa_circuit, mapped_observables)])

# Use the job ID to retrieve your job data later
print(f">>> Job ID: {job.job_id()}")

>>> Job ID: d5k96q4jt3vs73ds5tgg


After a job is submitted, you can wait until either the job is completed within your current python instance, or use the `job_id` to retrieve the data at a later time.  (See the [section on retrieving jobs](/docs/guides/monitor-job#retrieve-job-results-at-a-later-time) for details.)

After the job completes, examine its output through the job's `result()` attribute.

In [5]:
# This is the result of the entire submission.  You submitted one Pub,
# so this contains one inner result (and some metadata of its own).
job_result = job.result()

# This is the result from our single pub, which had six observables,
# so contains information on all six.
pub_result = job.result()[0]

In [6]:
# Check there are six observables.
# If not, edit the comments in the previous cell and update this test.
assert len(pub_result.data.evs) == 6

<Admonition type="note" title="Alternative: run the example using a simulator">
  When you run your quantum program on a real device, your workload must wait in a queue before it runs. To save time, you can instead use the following code to run this small workload on the [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) with the Qiskit Runtime local testing mode. Note that this is only possible for a small circuit. When you scale up in the next section, you will need to use a real device.

  ```python

  # Use the following code instead if you want to run on a simulator:

  from qiskit_ibm_runtime.fake_provider import FakeBelemV2
  backend = FakeBelemV2()
  estimator = Estimator(backend)

  # Convert to an ISA circuit and layout-mapped observables.

  pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
  isa_circuit = pm.run(qc)
  mapped_observables = [
      observable.apply_layout(isa_circuit.layout) for observable in observables
  ]

  job = estimator.run([(isa_circuit, mapped_observables)])
  result = job.result()

  # This is the result of the entire submission.  You submitted one Pub,
  # so this contains one inner result (and some metadata of its own).

  job_result = job.result()

  # This is the result from our single pub, which had five observables,
  # so contains information on all five.

  pub_result = job.result()[0]
  ```
</Admonition>

### Step 4. Analyze the results

The analyze step is typically where you might post-process your results using, for example, measurement error mitigation or zero noise extrapolation (ZNE). You might feed these results into another workflow for further analysis or prepare a plot of the key values and data. In general, this step is specific to your problem.  For this example, plot each of the expectation values that were measured for our circuit.

The expectation values and standard deviations for the observables you specified to Estimator are accessed through the job result's `PubResult.data.evs` and `PubResult.data.stds` attributes. To obtain the results from Sampler, use the `PubResult.data.meas.get_counts()` function, which will return a `dict` of measurements in the form of bitstrings as keys and counts as their corresponding values. For more information, see [Get started with Sampler.](/docs/guides/get-started-with-primitives#get-started-with-sampler)

In [ ]:
# Plot the result

values = pub_result.data.evs

errors = pub_result.data.stds

# plotting graph
plt.plot(observables_labels, values, "-o")
plt.xlabel("Observables")
plt.ylabel("Values")
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg" alt="Output of the previous code cell" />

> **Note:** Když spustíš svůj kvantový program na skutečném zařízení, tvoje úloha musí čekat ve frontě, než se spustí. Abys ušetřil(a) čas, můžeš místo toho použít následující kód a spustit tuto malou úlohu na [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) s lokálním testovacím režimem Qiskit Runtime. Vezmi na vědomí, že to je možné pouze pro malý Circuit. Až v další části budeš škálovat, budeš potřebovat skutečné zařízení.
> 
>   ```python
> 
>   # Use the following code instead if you want to run on a simulator:
> 
>   from qiskit_ibm_runtime.fake_provider import FakeBelemV2
>   backend = FakeBelemV2()
>   estimator = Estimator(backend)
> 
>   # Convert to an ISA circuit and layout-mapped observables.
> 
>   pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
>   isa_circuit = pm.run(qc)
>   mapped_observables = [
>       observable.apply_layout(isa_circuit.layout) for observable in observables
>   ]
> 
>   job = estimator.run([(isa_circuit, mapped_observables)])
>   result = job.result()
> 
>   # This is the result of the entire submission.  You submitted one Pub,
>   # so this contains one inner result (and some metadata of its own).
> 
>   job_result = job.result()
> 
>   # This is the result from our single pub, which had five observables,
>   # so contains information on all five.
> 
>   pub_result = job.result()[0]
>   ```
### Krok 4. Analyzuj výsledky
Krok analýzy je typicky fáze, kdy zpracováváš výsledky dodatečně — například pomocí zmírnění chyb při měření nebo extrapolace nulového šumu (ZNE). Tyto výsledky můžeš předat do jiného pracovního postupu k dalšímu rozboru nebo připravit graf klíčových hodnot a dat. Obecně je tento krok specifický pro tvůj problém. V tomto příkladu vykresli každou ze středních hodnot, které byly pro náš Circuit naměřeny.

Střední hodnoty a směrodatné odchylky pro pozorovatelné veličiny, které jsi zadal(a) do Estimator, jsou přístupné přes atributy `PubResult.data.evs` a `PubResult.data.stds` výsledku úlohy. Pro získání výsledků ze Sampler použij funkci `PubResult.data.meas.get_counts()`, která vrátí `dict` měření ve formě bitových řetězců jako klíčů a počtů jako odpovídajících hodnot. Více informací najdeš v [Začínáme se Sampler.](/guides/get-started-with-primitives#get-started-with-sampler)

In [8]:
# Make sure the results follow the claim from the previous markdown cell.
# This can happen when the device occasionally behaves strangely. If this cell
# fails, you may just need to run the notebook again.

_results = {obs: val for obs, val in zip(observables_labels, values)}
for _label in ["IZ", "IX", "ZI", "XI"]:
    assert abs(_results[_label]) < 0.2
for _label in ["XX", "ZZ"]:
    assert _results[_label] > 0.8

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg)

Všimni si, že pro Qubit 0 a 1 jsou nezávislé střední hodnoty jak X, tak Z rovny 0, zatímco korelace (`XX` a `ZZ`) jsou rovny 1. To je charakteristický znak kvantového provázání.

In [ ]:
def get_qc_for_n_qubit_GHZ_state(n: int) -> QuantumCircuit:
    """This function will create a qiskit.QuantumCircuit (qc) for an n-qubit GHZ state.

    Args:
        n (int): Number of qubits in the n-qubit GHZ state

    Returns:
        QuantumCircuit: Quantum circuit that generate the n-qubit GHZ state, assuming all qubits start in the 0 state
    """
    if isinstance(n, int) and n >= 2:
        qc = QuantumCircuit(n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
    else:
        raise Exception("n is not a valid input")
    return qc


# Create a new circuit with two qubits (first argument) and two classical
# bits (second argument)
n = 100
qc = get_qc_for_n_qubit_GHZ_state(n)

## Škálování na velký počet Qubitů
V oblasti kvantových výpočtů je práce na úrovni utility klíčová pro pokrok v oboru. Taková práce vyžaduje výpočty ve výrazně větším měřítku – s Circuit, které mohou využívat více než 100 Qubitů a přes 1000 Gate. Tento příklad ukazuje, jak lze provádět práci na úrovni utility na IBM&reg; QPU tím, že vytvoříš a analyzuješ 100-qubitový stav GHZ. Využívá postup Qiskit patterns a na závěr měří střední hodnotu $\langle Z_0 Z_i \rangle$ pro každý Qubit.

### Krok 1. Mapování problému
Napiš funkci, která vrátí `QuantumCircuit` připravující $n$-qubitový stav GHZ (v podstatě rozšířený Bellův stav), pak tuto funkci použij k přípravě 100-qubitového stavu GHZ a shromáždi Observable, která chceš měřit.

In [ ]:
# ZZII...II, ZIZI...II, ... , ZIII...IZ
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]
print(operator_strings)
print(len(operator_strings))

operators = [SparsePauliOp(operator) for operator in operator_strings]

['ZZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII

Dále namapuj operátory, které tě zajímají. Tento příklad používá operátory `ZZ` mezi Qubity, aby zkoumal chování při jejich vzrůstající vzdálenosti. Čím méně přesné (zkorumpované) střední hodnoty mezi vzdálenými Qubity odhalí míru přítomného šumu.

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(
    simulator=False, operational=True, min_num_qubits=100
)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

isa_circuit = pm.run(qc)
isa_operators_list = [op.apply_layout(isa_circuit.layout) for op in operators]

### Step 3. Execute on hardware

Submit the job and enable error suppression by using a technique to reduce errors called [dynamical decoupling.](../api/qiskit-ibm-runtime/options-dynamical-decoupling-options) The resilience level specifies how much resilience to build against errors. Higher levels generate more accurate results, at the expense of longer processing times.  For further explanation of the options set in the following code, see [Configure error mitigation for Qiskit Runtime.](/docs/guides/configure-error-mitigation)

In [ ]:
options = EstimatorOptions()
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"

# Create an Estimator object
estimator = Estimator(backend, options=options)

In [13]:
# Submit the circuit to Estimator
job = estimator.run([(isa_circuit, isa_operators_list)])
job_id = job.job_id()
print(job_id)

d5k9mmqvcahs73a1ni3g


### Krok 3. Spuštění na hardwaru
Odešli úlohu a aktivuj potlačení chyb pomocí techniky zvané [dynamical decoupling.](../api/qiskit-ibm-runtime/options-dynamical-decoupling-options) Úroveň odolnosti určuje míru ochrany proti chybám. Vyšší úrovně generují přesnější výsledky, ale za cenu delší doby zpracování. Podrobnější vysvětlení možností nastavených v následujícím kódu najdeš v části [Konfigurace zmírnění chyb pro Qiskit Runtime.](/guides/configure-error-mitigation)

In [ ]:
# data
data = list(range(1, len(operators) + 1))  # Distance between the Z operators
result = job.result()[0]
values = result.data.evs  # Expectation value at each Z operator.
values = [
    v / values[0] for v in values
]  # Normalize the expectation values to evaluate how they decay with distance.

# plotting graph
plt.plot(data, values, marker="o", label="100-qubit GHZ state")
plt.xlabel("Distance between qubits $i$")
plt.ylabel(r"$\langle Z_i Z_0 \rangle / \langle Z_1 Z_0 \rangle $")
plt.legend()
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/de91ebd0-0.svg" alt="Output of the previous code cell" />

The previous plot shows that as the distance between qubits increases, the signal decays because of the presence of noise.

## Next steps

<Admonition type="tip" title="Recommendations">
  -   Try one of these tutorials:
      - [Ground-state energy estimation of the Heisenberg chain with VQE](/docs/tutorials/spin-chain-vqe)
      - Solve optimization problems using [QAOA](/docs/tutorials/quantum-approximate-optimization-algorithm)
      - Train [quantum kernel](/docs/tutorials/quantum-kernel-training) models for machine learning tasks
  - Find detailed installation instructions in the [Install Qiskit](/docs/guides/install-qiskit) guide.
  - If you prefer not to install Qiskit locally, read about options to use Qiskit in an [online development environment.](/docs/guides/online-lab-environments)
  - To save multiple account credentials or to specify other account options, see detailed instructions in the [Save your login credentials](/docs/guides/save-credentials#save-your-access-credentials) guide.
</Admonition>